# When Should We Trust Market Probabilities?

This notebook is the presentation layer for the reproducible audit pipeline. It reads generated artifacts and does not download data, engineer features, fit models, or choose the final recommendation.

The project tests whether normalized market-average closing probabilities should be used as-is, recalibrated, or augmented with public pre-match information.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

metrics = json.loads((ROOT / "results/audit_metrics.json").read_text())
slices = pd.read_csv(ROOT / "results/slice_metrics.csv")
decision = metrics["decision"]["recommendation"]
display(Markdown(f"## Decision: `{decision}`"))

## Four expanding-window tests

Each model is trained only on earlier seasons and evaluated on the next complete season. Multiclass log loss is the primary metric; lower values are better.

In [ ]:
records = pd.DataFrame(metrics["metric_records"])
comparison = records[records["metric"].isin(["log_loss", "brier", "accuracy", "macro_f1"])]
display(
    comparison.pivot_table(index=["test_season", "model"], columns="metric", values="value").round(
        4
    )
)
rolling_figure = (ROOT / "results/figures/rolling_log_loss.png").as_posix()
display(Markdown(f"![Out-of-time log loss by test season]({rolling_figure})"))

## Calibration and model-risk finding

Raw market probabilities are already closely calibrated in the pooled out-of-time sample. Recalibration does not deliver a robust improvement, while adding Elo, form, rest, league, and season progress worsens log loss in all four test seasons.

In [ ]:
calibration_figure = (ROOT / "results/figures/calibration_by_class.png").as_posix()
display(Markdown(f"![Pooled calibration by outcome class]({calibration_figure})"))
display(
    slices[(slices["slice_type"] == "league") & (slices["model"] == "market_raw")].sort_values(
        "log_loss"
    )
)

## Scope and limitations

**This analysis supports:** raw market probabilities were closely calibrated in the tested sample; the selected recalibration did not improve robustly; and the selected public pre-match features worsened performance.

**This analysis does not establish:** that all football markets or vendor models are efficient, that richer proprietary data have no value, or that no profitable strategy can exist. The original team notebook is not distributed, and its leakage-affected metrics are not reused.